# 15-coin log return embeddings for new CMC data

This notebook follows the original `1.data_process.ipynb` logic for the new 15-coin price file.

Pipeline:
1. Read the new CMC price workbook.
2. Compute log returns for all 15 cryptocurrencies.
3. Build time-delay embeddings for all parameter combinations:
   - window size `m in {7, 14, 21, 28}`
   - time lag `tau in {1, 2, 3}`
4. Save 12 embedding results to `8. New data/15coin_log_return_embeddings`.


In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_FILE = Path("/Users/jane/Documents/202511吾-Systems/8. New data/cmc_0321_加密货币市场规模前15资产价格.xlsx")
OUTPUT_DIR = Path("/Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings")

WINDOW_SIZES = [7, 14, 21, 28]
LAGS = [1, 2, 3]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file exists: {INPUT_FILE.exists()}")
print(f"Output directory: {OUTPUT_DIR}")


Input file exists: True
Output directory: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings


In [ ]:
def time_delay_embedding(series, window_size, lag):
    """Create a sliding-window time-delay embedding for one series."""
    series = np.asarray(series, dtype=float)
    n_obs = len(series)
    n_rows = n_obs - (window_size - 1) * lag

    if n_rows <= 0:
        raise ValueError(
            f"Series is too short for window_size={window_size}, lag={lag}."
        )

    embedded = np.empty((n_rows, window_size), dtype=float)
    for i in range(n_rows):
        embedded[i, :] = series[i:(i + window_size * lag):lag]
    return embedded


def load_price_data(path):
    df = pd.read_excel(path)
    if "Date" not in df.columns:
        raise KeyError("The input file must contain a 'Date' column.")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates(subset="Date").set_index("Date")
    df = df.apply(pd.to_numeric, errors="coerce")

    if df.isna().any().any():
        bad_cols = df.columns[df.isna().any()].tolist()
        raise ValueError(f"Missing or non-numeric values found in columns: {bad_cols}")

    if (df <= 0).any().any():
        bad_cols = df.columns[(df <= 0).any()].tolist()
        raise ValueError(f"Non-positive values found in columns: {bad_cols}")

    return df


def build_embedding_panel(df_returns, window_size, lag):
    coins = df_returns.columns.to_list()
    embedded_by_coin = [
        time_delay_embedding(df_returns[coin].to_numpy(), window_size=window_size, lag=lag)
        for coin in coins
    ]

    panel = np.stack(embedded_by_coin, axis=1)
    dates = df_returns.index[(window_size - 1) * lag :]

    if len(dates) != panel.shape[0]:
        raise RuntimeError("Date alignment failed when building the embedding panel.")

    return panel, coins, dates


def save_embedding_result(output_file, panel, coins, dates, window_size, lag):
    np.savez_compressed(
        output_file,
        embedded_array=panel,
        coins=np.array(coins, dtype=str),
        dates=dates.strftime("%Y-%m-%d").to_numpy(dtype=str),
        window_size=np.int64(window_size),
        lag=np.int64(lag),
        n_time=np.int64(panel.shape[0]),
        n_coins=np.int64(panel.shape[1]),
        embedding_dim=np.int64(panel.shape[2]),
        value_type=np.array("log_return")
    )


In [ ]:
price_df = load_price_data(INPUT_FILE)
log_return_df = np.log(price_df / price_df.shift(1)).dropna()

log_return_path = OUTPUT_DIR / "cmc15_log_returns.csv"
log_return_df.reset_index().to_csv(log_return_path, index=False)

print(f"Price data shape: {price_df.shape}")
print(f"Log-return data shape: {log_return_df.shape}")
print(f"Coins ({len(price_df.columns)}): {price_df.columns.tolist()}")
print(f"Log returns saved to: {log_return_path}")

log_return_df.head()


Price data shape: (1874, 15)
Log-return data shape: (1873, 15)
Coins (15): ['ETH', 'BTC', 'XRP', 'USDT', 'USDC', 'BNB', 'DOGE', 'ADA', 'TRX', 'SOL', 'LTC', 'IOTA', 'BCH', 'XTZ', 'XLM']
Log returns saved to: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings/cmc15_log_returns.csv


,ETH,BTC,XRP,USDT,USDC,BNB,DOGE,ADA,TRX,SOL,LTC,IOTA,BCH,XTZ,XLM
Date,,,,,,,,,,,,,,,
2020-08-11,-0.042088,-0.041077,-0.038278,-0.000878,0.000134,-0.068376,-0.042914,-0.045865,-0.065916,0.053352,-0.074253,-0.038635,-0.070447,-0.059492,-0.059126
2020-08-12,0.021133,0.015924,-0.004382,0.000102,-0.000201,0.022284,0.025717,-0.003510,0.003503,0.132051,0.007733,0.044629,0.010027,0.083795,0.013094
2020-08-13,0.092655,0.016955,0.040654,0.001062,0.000294,0.009957,0.005700,0.021108,0.099721,-0.003662,0.046937,-0.020473,0.038184,-0.052954,-0.003554
2020-08-14,0.030777,0.000082,0.021185,0.000110,-0.000053,0.054650,0.039339,-0.007444,0.104583,-0.096814,-0.006320,0.077285,-0.008046,-0.006355,0.035749
2020-08-15,-0.012560,0.007826,-0.002037,-0.000857,-0.000135,-0.002007,-0.017868,-0.002157,0.016124,-0.068459,0.052657,0.028791,0.029521,-0.010248,0.021524


In [ ]:
manifest_rows = []

for window_size, lag in product(WINDOW_SIZES, LAGS):
    panel, coins, dates = build_embedding_panel(log_return_df, window_size=window_size, lag=lag)
    output_file = OUTPUT_DIR / f"cmc15_log_return_embedding_m{window_size:02d}_tau{lag}.npz"
    save_embedding_result(output_file, panel, coins, dates, window_size, lag)

    manifest_rows.append(
        {
            "output_file": output_file.name,
            "window_size": window_size,
            "lag": lag,
            "n_time": panel.shape[0],
            "n_coins": panel.shape[1],
            "embedding_dim": panel.shape[2],
            "first_date": dates[0].strftime("%Y-%m-%d"),
            "last_date": dates[-1].strftime("%Y-%m-%d")
        }
    )

    print(
        f"Saved {output_file.name} | shape={panel.shape} | "
        f"date range={dates[0].date()} to {dates[-1].date()}"
    )

manifest_df = pd.DataFrame(manifest_rows).sort_values(["window_size", "lag"]).reset_index(drop=True)
manifest_path = OUTPUT_DIR / "cmc15_log_return_embedding_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"\nManifest saved to: {manifest_path}")
manifest_df


Saved cmc15_log_return_embedding_m07_tau1.npz | shape=(1867, 15, 7) | date range=2020-08-17 to 2025-09-26


Saved cmc15_log_return_embedding_m07_tau2.npz | shape=(1861, 15, 7) | date range=2020-08-23 to 2025-09-26
Saved cmc15_log_return_embedding_m07_tau3.npz | shape=(1855, 15, 7) | date range=2020-08-29 to 2025-09-26
Saved cmc15_log_return_embedding_m14_tau1.npz | shape=(1860, 15, 14) | date range=2020-08-24 to 2025-09-26
Saved cmc15_log_return_embedding_m14_tau2.npz | shape=(1847, 15, 14) | date range=2020-09-06 to 2025-09-26
Saved cmc15_log_return_embedding_m14_tau3.npz | shape=(1834, 15, 14) | date range=2020-09-19 to 2025-09-26
Saved cmc15_log_return_embedding_m21_tau1.npz | shape=(1853, 15, 21) | date range=2020-08-31 to 2025-09-26
Saved cmc15_log_return_embedding_m21_tau2.npz | shape=(1833, 15, 21) | date range=2020-09-20 to 2025-09-26


Saved cmc15_log_return_embedding_m21_tau3.npz | shape=(1813, 15, 21) | date range=2020-10-10 to 2025-09-26


Saved cmc15_log_return_embedding_m28_tau1.npz | shape=(1846, 15, 28) | date range=2020-09-07 to 2025-09-26
Saved cmc15_log_return_embedding_m28_tau2.npz | shape=(1819, 15, 28) | date range=2020-10-04 to 2025-09-26
Saved cmc15_log_return_embedding_m28_tau3.npz | shape=(1792, 15, 28) | date range=2020-10-31 to 2025-09-26

Manifest saved to: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings/cmc15_log_return_embedding_manifest.csv


,output_file,window_size,lag,n_time,n_coins,embedding_dim,first_date,last_date
0,cmc15_log_return_embedding_m07_tau1.npz,7,1,1867,15,7,2020-08-17,2025-09-26
1,cmc15_log_return_embedding_m07_tau2.npz,7,2,1861,15,7,2020-08-23,2025-09-26
2,cmc15_log_return_embedding_m07_tau3.npz,7,3,1855,15,7,2020-08-29,2025-09-26
3,cmc15_log_return_embedding_m14_tau1.npz,14,1,1860,15,14,2020-08-24,2025-09-26
4,cmc15_log_return_embedding_m14_tau2.npz,14,2,1847,15,14,2020-09-06,2025-09-26
5,cmc15_log_return_embedding_m14_tau3.npz,14,3,1834,15,14,2020-09-19,2025-09-26
6,cmc15_log_return_embedding_m21_tau1.npz,21,1,1853,15,21,2020-08-31,2025-09-26
7,cmc15_log_return_embedding_m21_tau2.npz,21,2,1833,15,21,2020-09-20,2025-09-26
8,cmc15_log_return_embedding_m21_tau3.npz,21,3,1813,15,21,2020-10-10,2025-09-26
9,cmc15_log_return_embedding_m28_tau1.npz,28,1,1846,15,28,2020-09-07,2025-09-26


In [ ]:
sample_file = OUTPUT_DIR / "cmc15_log_return_embedding_m21_tau1.npz"
sample = np.load(sample_file, allow_pickle=True)

print("NPZ keys:", sample.files)
print("embedded_array shape:", sample["embedded_array"].shape)
print("coins:", sample["coins"].tolist())
print("first 3 dates:", sample["dates"][:3].tolist())


NPZ keys: ['embedded_array', 'coins', 'dates', 'window_size', 'lag', 'n_time', 'n_coins', 'embedding_dim', 'value_type']
embedded_array shape: (1853, 15, 21)
coins: ['ETH', 'BTC', 'XRP', 'USDT', 'USDC', 'BNB', 'DOGE', 'ADA', 'TRX', 'SOL', 'LTC', 'IOTA', 'BCH', 'XTZ', 'XLM']
first 3 dates: ['2020-08-31', '2020-09-01', '2020-09-02']
